In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdMolDescriptors import GetUSRCAT
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# ================== 1. 读取数据 ==================
df = pd.read_csv("molecules.csv")  # 文件需含 'SMILES' 和 'Label' 列
df.columns = df.columns.str.replace('[^0-9a-zA-Z_]', '_', regex=True)  # 清理列名

# 转换SMILES为RDKit分子对象
df['Mol'] = df['SMILES'].apply(Chem.MolFromSmiles)

# 过滤掉无效分子
df = df[df['Mol'].notnull()].reset_index(drop=True)

print(f"有效分子数（用于Morgan指纹）：{len(df)}")

# ================== 2. 计算Morgan指纹 ==================
print("生成 Morgan 指纹中...")
morgan_fps = [AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=1024) for m in df['Mol']]
morgan_array = np.array([np.array(fp) for fp in morgan_fps])  # 转成numpy数组

# ================== 3. 生成3D构象 ==================
print("生成 3D 构象中...")
def generate_3d_conformer(mol):
    mol = Chem.AddHs(mol)  # 加氢
    try:
        AllChem.EmbedMolecule(mol, randomSeed=42, maxAttempts=500)
        AllChem.UFFOptimizeMolecule(mol)
        return mol
    except:
        return None

df['Mol3D'] = df['Mol'].apply(generate_3d_conformer)

# 过滤无效3D构象分子，单独存储
df_3d = df[df['Mol3D'].notnull()].reset_index(drop=True)
print(f"有效3D构象分子数（用于USRCAT指纹）：{len(df_3d)}")

# ================== 4. 计算USRCAT指纹 ==================
print("计算 USRCAT 指纹中（跳过无效分子）...")

def safe_get_usrcat(mol):
    if mol is None or mol.GetNumAtoms() < 3:
        return None
    try:
        return GetUSRCAT(mol)
    except:
        return None

valid_usrcat = []
valid_smiles = []
valid_labels = []

for mol3d, row in zip(df_3d['Mol3D'], df_3d.iterrows()):
    vec = safe_get_usrcat(mol3d)
    if vec is not None:
        valid_usrcat.append(vec)
        valid_smiles.append(row[1]['SMILES'])
        valid_labels.append(row[1]['Label'])

usrcat_array = np.array(valid_usrcat)
df_usrcat = pd.DataFrame({'SMILES': valid_smiles, 'Label': valid_labels})

print(f"成功生成 USRCAT 指纹的分子数：{len(df_usrcat)}")

# ================== 5. t-SNE 降维 ==================
print("执行 t-SNE 降维中...")

# 对 Morgan 指纹进行 t-SNE 降维
tsne_morgan = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
morgan_tsne = tsne_morgan.fit_transform(morgan_array)
df['Morgan_tSNE1'] = morgan_tsne[:, 0]
df['Morgan_tSNE2'] = morgan_tsne[:, 1]

# 对 USRCAT 指纹进行 t-SNE 降维
tsne_usrcat = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
usrcat_tsne = tsne_usrcat.fit_transform(usrcat_array)
df_usrcat['USRCAT_tSNE1'] = usrcat_tsne[:, 0]
df_usrcat['USRCAT_tSNE2'] = usrcat_tsne[:, 1]
# ================== 6. 导出 t-SNE 数据用于 Origin 绘图 ==================

# 导出 Morgan t-SNE 数据
df_morgan_out = df[['SMILES', 'Label', 'Morgan_tSNE1', 'Morgan_tSNE2']]
df_morgan_out.to_csv("tSNE_Morgan_Output.csv", index=False)

# 导出 USRCAT t-SNE 数据
df_usrcat_out = df_usrcat[['SMILES', 'Label', 'USRCAT_tSNE1', 'USRCAT_tSNE2']]
df_usrcat_out.to_csv("tSNE_USRCAT_Output.csv", index=False)

print("✅ 已成功导出 t-SNE 降维数据，您可以在 Origin 中绘图。")
print("- tSNE_Morgan_Output.csv")
print("- tSNE_USRCAT_Output.csv")


有效分子数（用于Morgan指纹）：3169
生成 Morgan 指纹中...
生成 3D 构象中...


[23:30:22] UFFTYPER: Unrecognized charge state for atom: 8
[23:30:22] UFFTYPER: Unrecognized charge state for atom: 8
[23:30:22] UFFTYPER: Unrecognized charge state for atom: 5
[23:30:22] UFFTYPER: Unrecognized charge state for atom: 5
[23:30:32] UFFTYPER: Unrecognized charge state for atom: 2
[23:30:32] UFFTYPER: Unrecognized atom type: Pb3+3 (2)
[23:30:32] UFFTYPER: Unrecognized charge state for atom: 2
[23:30:32] UFFTYPER: Unrecognized atom type: Pb3+3 (2)
[23:30:41] UFFTYPER: Unrecognized atom type: Co3+3 (0)
[23:30:41] UFFTYPER: Unrecognized atom type: Co3+3 (0)
[23:30:48] UFFTYPER: Unrecognized charge state for atom: 6
[23:30:48] UFFTYPER: Unrecognized charge state for atom: 6
[23:30:56] UFFTYPER: Warning: hybridization set to SP for atom 1
[23:30:56] UFFTYPER: Unrecognized charge state for atom: 1
[23:30:56] UFFTYPER: Warning: hybridization set to SP for atom 1
[23:30:56] UFFTYPER: Unrecognized charge state for atom: 1
[23:31:05] UFFTYPER: Unrecognized charge state for atom: 0
[

有效3D构象分子数（用于USRCAT指纹）：3168
计算 USRCAT 指纹中（跳过无效分子）...
成功生成 USRCAT 指纹的分子数：3159
执行 t-SNE 降维中...


/home/rdkit/anaconda3/lib/python3.9/site-packages/sklearn/manifold/_t_sne.py:982: FutureWarning: The PCA initialization in TSNE will change to have the standard deviation of PC1 equal to 1e-4 in 1.2. This will ensure better convergence.
  warnings.warn(
/home/rdkit/anaconda3/lib/python3.9/site-packages/sklearn/manifold/_t_sne.py:982: FutureWarning: The PCA initialization in TSNE will change to have the standard deviation of PC1 equal to 1e-4 in 1.2. This will ensure better convergence.
  warnings.warn(


✅ 已成功导出 t-SNE 降维数据，您可以在 Origin 中绘图。
- tSNE_Morgan_Output.csv
- tSNE_USRCAT_Output.csv
